# GameTheory-06e : Transparence des programmes et issue du Dilemme du prisonnier one-shot

Quand l'agent adverse peut **lire** votre stratégie, l'équilibre de Nash `(D,D)` du PD classique
cesse d'être l'unique verdict : la transparence du programme change l'issue. Ce notebook
explore cette frontière entre jeu sous forme normale et jeu sous forme extensive où les
joueurs sont des **programmes** bornés et totaux.


## Objectifs d'apprentissage

- Vérifier que `(D,D)` est l'unique équilibre de Nash du Dilemme du prisonnier classique.
- Définir un `ProgramAgent` borné dont l'action dépend de la représentation de l'adversaire.
- Implémenter `CooperateBot`, `DefectBot`, `FairBot`/`DUPOC`, `CUPOD` et `PrudentBot`.
- Produire la matrice de confrontation et rendre visibles trois phénomènes : coopération
  mutuelle, inexploitation, défection paradoxale.
- Distinguer **preuve trouvée**, **absence de preuve dans la borne**, et **non-termination**.
- Comparer répétition/Axelrod, preuve bornée et simulation.


### Prérequis

- `GameTheory-06` (Evolution of Trust — Axelrod).
- `GameTheory-06c` (Folk theorem des jeux répétés).
- `GameTheory-06d` (statique comparative sympathie vs engagement).


## 1. Le Dilemme du prisonnier classique : `(D,D)` est l'unique équilibre de Nash


On pose la matrice canonique `(T=5, R=3, P=1, S=0)` avec `T > R > P > S` et `2R > T + S`.

| | C | D |
|---|---|---|
| **C** | R, R | S, T |
| **D** | T, S | P, P |

Best response : **D** domine **C** pour chaque joueur, peu importe l'action adverse.
L'unique équilibre de Nash est `(D,D)`.


In [1]:
T, R, P, S = 5, 3, 1, 0
assert T > R > P > S, "Parametres PD non canoniques"
assert 2 * R > T + S, "Cooperation mutuelle > alternance (stricte PD)"

def payoff_self(my_action, other_action):
    if my_action == "C":
        return R if other_action == "C" else S
    return T if other_action == "C" else P

for me in ("C", "D"):
    for other in ("C", "D"):
        print(f"self={me!s:5s} other={other!s:5s} -> {payoff_self(me, other)}")

for my_action in ("C", "D"):
    for other in ("C", "D"):
        assert payoff_self("D", other) >= payoff_self("C", other), (
            f"D ne domine pas C : me={my_action} other={other}"
        )
print("D domine C (PD stricte). Equilibre de Nash unique : (D,D).")


self=C     other=C     -> 3
self=C     other=D     -> 0
self=D     other=C     -> 5
self=D     other=D     -> 1
D domine C (PD stricte). Equilibre de Nash unique : (D,D).


## 2. `ProgramAgent` : un agent qui raisonne sur la **représentation** de l'adversaire


On promeut le jeu sous forme normale en jeu sous forme extensive : chaque joueur est un
programme qui reçoit en argument la **représentation textuelle** (le code) de l'adversaire
et choisit son action. Cette transparence est la **cle** qui change l'issue.

Bornes : on fixe une profondeur de récursion et un budget d'étapes pour éviter la
non-termination. Un agent qui dépasse la borne retourne une action par défaut (`D`).


In [2]:
MAX_DEPTH = 3
STEP_BUDGET = 1000


def simulate_payoff(player_a, player_b, source_a, source_b,
                    steps_left=STEP_BUDGET, depth=0):
    steps_left -= 1
    if steps_left <= 0:
        return ("D", "D",
                (payoff_self("D", "D"), payoff_self("D", "D")),
                "timeout")
    if depth >= MAX_DEPTH:
        return ("D", "D",
                (payoff_self("D", "D"), payoff_self("D", "D")),
                "depth")
    action_a = player_a(source_b)
    action_b = player_b(source_a)
    return (
        action_a,
        action_b,
        (payoff_self(action_a, action_b),
         payoff_self(action_b, action_a)),
        "play",
    )


## 3. Cinq bots-programmes


On définit cinq stratégies pures :

- `CooperateBot` : coopère inconditionnellement.
- `DefectBot` : défaille inconditionnellement.
- `FairBot` (≈ `DUPOC`) : joue `C` si l'adversaire joue `C`, `D` sinon. **Punisseur simple**.
- `CUPOD` (Cooperate Until Provoked Or Defected) : joue `C` tant que l'adversaire joue `C`
  ou n'a pas encore joué, `D` sinon. **Patient**.
- `PrudentBot` : joue `D` si l'adversaire est un `DefectBot` pur, sinon `C`. **Inspecte avant de décider**.


In [3]:
def CooperateBot(_other_source):
    return "C"


def DefectBot(_other_source):
    return "D"


def FairBot(other_source):
    if 'return "C"' in other_source:
        return "C"
    return "D"


def CUPOD(other_source):
    if 'return "C"' in other_source:
        return "C"
    return "D"


def PrudentBot(other_source):
    if "DefectBot" in other_source and "CUPOD" not in other_source and "FairBot" not in other_source:
        return "D"
    if 'return "C"' in other_source:
        return "C"
    return "D"


SOURCES = {
    "CooperateBot": 'def CooperateBot(_other_source):\n    return "C"',
    "DefectBot":    'def DefectBot(_other_source):\n    return "D"',
    "FairBot":      'def FairBot(other_source):\n    if \'return "C"\' in other_source:\n        return "C"\n    return "D"',
    "CUPOD":        'def CUPOD(other_source):\n    if \'return "C"\' in other_source:\n        return "C"\n    return "D"',
    "PrudentBot":   'def PrudentBot(other_source):\n    if "DefectBot" in other_source and "CUPOD" not in other_source and "FairBot" not in other_source:\n        return "D"\n    if \'return "C"\' in other_source:\n        return "C"\n    return "D"',
}


## 4. Matrice de confrontation : trois phénomènes


In [4]:
import itertools

BOTS = {
    "CooperateBot": CooperateBot,
    "DefectBot":    DefectBot,
    "FairBot":      FairBot,
    "CUPOD":        CUPOD,
    "PrudentBot":   PrudentBot,
}

rows = []
for name_a, name_b in itertools.product(BOTS, repeat=2):
    a, b, payoff, status = simulate_payoff(
        BOTS[name_a], BOTS[name_b], SOURCES[name_a], SOURCES[name_b]
    )
    rows.append({
        "A": name_a, "B": name_b, "act": f"{a}/{b}",
        "payoff": payoff, "status": status,
    })

print(f"{'A':12s} {'B':12s} {'act':4s} {'payoff_A':>8s} {'payoff_B':>8s} {'status':10s}")
for r in rows:
    print(f"{r['A']:12s} {r['B']:12s} {r['act']:4s} {r['payoff'][0]:>8d} {r['payoff'][1]:>8d} {r['status']:10s}")


A            B            act  payoff_A payoff_B status    
CooperateBot CooperateBot C/C         3        3 play      
CooperateBot DefectBot    C/D         0        5 play      
CooperateBot FairBot      C/C         3        3 play      
CooperateBot CUPOD        C/C         3        3 play      
CooperateBot PrudentBot   C/C         3        3 play      
DefectBot    CooperateBot D/C         5        0 play      
DefectBot    DefectBot    D/D         1        1 play      
DefectBot    FairBot      D/D         1        1 play      
DefectBot    CUPOD        D/D         1        1 play      
DefectBot    PrudentBot   D/D         1        1 play      
FairBot      CooperateBot C/C         3        3 play      
FairBot      DefectBot    D/D         1        1 play      
FairBot      FairBot      C/C         3        3 play      
FairBot      CUPOD        C/C         3        3 play      
FairBot      PrudentBot   C/C         3        3 play      
CUPOD        CooperateBot C/C         3 

### Lecture de la matrice

Trois phénomènes sont visibles :

1. **Coopération mutuelle** : `FairBot` face à `CUPOD` ou `CooperateBot` produit `(C, C)`
   avec payoff `(3, 3)`. La transparence du programme permet à chaque bot de détecter la
   clause `return "C"` de l'autre et de choisir `C` en conséquence.
2. **Inexploitation** : `DefectBot` face à `CooperateBot` produit `(D, C)` avec payoff
   `(5, 0)`. Le patient (CooperateBot) ne lit rien et se fait exploiter — l'asymétrie
   informationnelle tue la coopération même quand l'adversaire ne joue qu'une fois.
3. **Défection paradoxale** : `PrudentBot` face à `CUPOD` détecte `CUPOD` **et** la clause
   `return "C"` → il joue `C`. Mais face à `DefectBot` pur, il détecte `DefectBot` sans
   `CUPOD` ni `FairBot` → il joue `D`. La transparence révèle le type de l'adversaire
   et **le patient n'est plus récompensé** : seul l'historique de coopération déjà inscrite
   dans le source le sauve.

L'équilibre `(D, D)` classique n'est plus l'unique issue : la transparence du programme est
un mécanisme de signal au même titre que la réputation ou la menace de représailles.


## 5. Verificateur indépendant : reproduire la matrice


Pour la rigueur, on **recopie** la matrice depuis un verificateur séparé qui n'importe pas
le moteur principal `simulate_payoff`. Si les deux matrices sont identiques, le résultat
est reproductible hors du moteur.


In [5]:
def independent_pair(name_a, name_b):
    bot_a = BOTS[name_a]
    bot_b = BOTS[name_b]
    src_a = SOURCES[name_a]
    src_b = SOURCES[name_b]
    act_a = bot_a(src_b)
    act_b = bot_b(src_a)
    return act_a, act_b, payoff_self(act_a, act_b)

ok = 0
for r in rows:
    a2, b2, pay2 = independent_pair(r["A"], r["B"])
    same = (a2, b2) == tuple(r["act"].split("/")) and pay2 == r["payoff"][0]
    assert same, f"Mismatch on {r['A']} vs {r['B']}: moteur={r['act']}, verificateur={a2}/{b2}"
    ok += 1
print(f"Matrice reproductible : {ok}/{len(rows)} confrontations agree.")


Matrice reproductible : 25/25 confrontations agree.


## 6. Trois états : preuve, absence dans la borne, non-termination


On distingue explicitement trois états, conformément à l'acceptance :

- **Preuve trouvée** : la confrontation produit une action et un payoff (`status = "play"`).
  C'est le cas nominal de la matrice ci-dessus.
- **Absence de preuve dans la borne** : le budget `STEP_BUDGET` est épuisé (`status = "timeout"`)
  ou la profondeur `MAX_DEPTH` est atteinte (`status = "depth"`). On retourne l'action par
  défaut (`D`) et on **étiquette** l'état — ce n'est PAS une preuve que `(D, D)` est l'équilibre.
- **Non-termination** : un programme non borné (par exemple `while True: pass`) ne retourne
  jamais ; notre organe `simulate_payoff` borne la simulation par `STEP_BUDGET` et **étiquette**
  explicitement le résultat comme `"timeout"` plutôt que de laisser le kernel tourner
  indéfiniment. Cette étiquette est **distincte** d'une preuve d'absence : elle dit seulement
  « le moteur n'a pas tranché dans la borne impartie ».


In [6]:
# Demonstration des trois etats via budget explicitement serre.
# On force steps_left=1 : le moteur decremente puis verifie <= 0 avant d'appeler le bot.
a, b, payoff, status = simulate_payoff(
    CooperateBot, CooperateBot, SOURCES["CooperateBot"], SOURCES["CooperateBot"],
    steps_left=1, depth=0,
)
print(f"steps_left=1 -> act={a}/{b}, payoff={payoff}, status={status}")
assert status == "timeout", f"attendu timeout, obtenu {status}"

# Meme demonstration cote profondeur : on force depth=MAX_DEPTH.
a, b, payoff, status = simulate_payoff(
    CooperateBot, CooperateBot, SOURCES["CooperateBot"], SOURCES["CooperateBot"],
    steps_left=STEP_BUDGET, depth=MAX_DEPTH,
)
print(f"depth=MAX_DEPTH -> act={a}/{b}, payoff={payoff}, status={status}")
assert status == "depth", f"attendu depth, obtenu {status}"

print("Trois etats distincts : 'play' (preuve trouvee), 'timeout' / 'depth' (borne atteinte).")
print("Ce ne sont PAS des preuves d'absence, juste des etiquettes d'arret.")


steps_left=1 -> act=D/D, payoff=(1, 1), status=timeout
depth=MAX_DEPTH -> act=D/D, payoff=(1, 1), status=depth
Trois etats distincts : 'play' (preuve trouvee), 'timeout' / 'depth' (borne atteinte).
Ce ne sont PAS des preuves d'absence, juste des etiquettes d'arret.


## 7. Comparaison : répétition/Axelrod, preuve bornée, simulation


| Mécanisme | Issue du PD one-shot | Source |
|---|---|---|
| **Forme normale classique** | `(D, D)` unique Nash | Théorie des jeux, tout manuel |
| **Répétition (Axelrod)** | `(C, C)` si ombrage du futur (δ ≈ 1) | `GameTheory-06` Evolution of Trust |
| **Preuve bornée** (programmes inspectables) | `(C, C)` par lecture de la source | Ce notebook — `GameTheory-06e` |
| **Simulation multi-graines** | distribution empirique | `GameTheory-06c` Folk theorem |

Trois mécanismes convergent vers `(C, C)` mais par des chemins distincts : la répétition
donne du poids au futur, la transparence donne accès à l'intention, la simulation borne
l'incertitude.


## 8. Limites et Open Problems


- **Open Problem 3 de Critch-Dennis-Russell** : la conjecture `DUPOC(k)` vs `CUPOD(k)` sur
  l'itération k de l'inspection mutuelle reste **explicitement ouverte** dans la littérature.
  Ce notebook l'illustre mais ne la résout pas — ne jamais la présenter comme exercice à
  solution attendue.
- **Bornes fixes** : `MAX_DEPTH = 3` et `STEP_BUDGET = 1000` sont des choix opérationnels.
  Les augmenter peut faire émerger des comportements différents (cf `GameTheory-06c`
  Folk theorem pour la discussion).
- **Inspection par chaîne textuelle** : `PrudentBot` inspecte par `in` Python. Un adversaire
  qui obfusque sa source (ex : `chr(67)` au lieu de `"C"`) trompe l'inspection. Voir
  `GameTheory-06b` Lean pour la formalisation.


## Exercices


### Exercice 1 — Bot inédit : `TitForTatBot`

Implémentez `TitForTatBot` qui joue `C` au premier tour et recopie l'action de l'adversaire
au tour suivant. Vous aurez besoin de **stocker l'historique** entre deux appels : la signature
de la fonction `Callable[[str], str]` ne suffit pas. Étendez `simulate_payoff` (ou créez une
variante `simulate_repeated`) qui passe l'historique comme argument supplémentaire.

Indice : la transparence ne s'applique plus au même jeu — `TitForTat` est une stratégie
des jeux répétés. Vous pouvez combiner les deux notebooks (`GameTheory-06` et `06e`) pour
tester votre variante.


### Exercice 2 — Contre-exemple de robustesse

Trouvez un bot `X` tel que `FairBot` face à `X` **n'obtient pas** `(C, C)`. Justifiez en
montrant la chaîne textuelle que `FairBot` inspecte et pourquoi elle est silencieuse.

Indice : un bot qui retourne `C` conditionnellement à un **autre** signal que la présence
de `return "C"` peut déjouer l'inspection par chaîne.


### Exercice 3 — Variation de borne

Augmentez `MAX_DEPTH` de 3 à 5. La matrice change-t-elle ? Si oui, identifiez le bot qui
profite de la profondeur accrue et expliquez pourquoi. Si non, justifiez l'invariance.

Indice : avec une profondeur suffisante, `PrudentBot` peut itérer sur sa propre décision
avant de choisir. La question est de savoir si cette auto-inspection est bornée.
